# AffectCare — Exploration Notebook

This notebook documents two pieces of real analysis behind the final model:

1. **The `n_mfcc=13` vs `n_mfcc=40` comparison** — why fewer coefficients generalized better
2. **The dataset audit** — how the siren/scream confusion in the `normal` class was found

Run this from the `notebooks/` folder with the `src/` folder importable (or copy it into `src/` to run directly).

## 1. MFCC coefficient comparison

The original pipeline used `n_mfcc=13`. Out of curiosity, I tested `n_mfcc=40` — more frequency detail per clip — expecting it to help. It didn't.

Below: extract both versions of the same file and visualize the difference in information density.

In [ ]:
import sys
sys.path.append("../src")

import librosa
import librosa.display
import matplotlib.pyplot as plt

sample_path = "../dataset/distress/"  # point this at any single distress .wav file
import os
sample_file = os.path.join(sample_path, os.listdir(sample_path)[0])

audio, sr = librosa.load(sample_file, sr=22050, mono=True)
audio_trimmed, _ = librosa.effects.trim(audio, top_db=18)

mfcc_13 = librosa.feature.mfcc(y=audio_trimmed, sr=sr, n_mfcc=13)
mfcc_40 = librosa.feature.mfcc(y=audio_trimmed, sr=sr, n_mfcc=40)

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

librosa.display.specshow(mfcc_13, x_axis="time", ax=axes[0])
axes[0].set_title("n_mfcc=13 — what the final model uses")

librosa.display.specshow(mfcc_40, x_axis="time", ax=axes[1])
axes[1].set_title("n_mfcc=40 — more detail, more room to overfit at this dataset size")

plt.tight_layout()
plt.show()

### What actually happened when I trained on each

| Setting | Final train loss | Test recall | Notes |
|---|---|---|---|
| `n_mfcc=13` | ~0.18–0.20 | 87–91% (stable across runs) | Smooth loss curve, consistent results |
| `n_mfcc=40` | ~0.05 (near zero) | 82.5% (worse) | Textbook overfitting — the model memorized training clips instead of learning the general shape of distress |

This is a direct, observed bias-variance tradeoff: more features gave the model more capacity to memorize noise rather than signal, at only ~800 training files. `n_mfcc=13` was kept as the final setting for this reason, not as a default assumption.

## 2. Dataset audit — finding the siren confusion

`src/audit_dataset.py` flags any `normal`-labeled clip with unusually high RMS energy — the idea being that a genuinely calm sound shouldn't have scream-level energy. Anything flagged is worth a manual listen.

In [ ]:
from audit_dataset import flag_suspicious_files

flagged = flag_suspicious_files("../dataset/normal", "normal", energy_threshold=0.15)

for filename, energy in sorted(flagged, key=lambda x: -x[1])[:10]:
    print(f"{filename:30s}  energy={energy}")

### The finding

Most flagged files were legitimately loud everyday sounds — chainsaws, vacuum cleaners, trains, clapping. But one cluster stood out clearly at the top of the energy ranking: **ESC-50's siren category (`-42` suffix)**, at 0.40–0.44 RMS energy — the highest of any flagged file.

A siren and a human scream share real acoustic similarity: both are sustained, high-pitched, high-energy bursts. This is a legitimate spurious-correlation risk — if the model leans on "loud + sustained + high-pitched" as its main signal for distress, an ambulance passing by could trigger a false alarm, and (more subtly) training on ambiguous siren-adjacent clips could blur the model's decision boundary.

This was left in the dataset intentionally rather than removed — a passing ambulance genuinely *shouldn't* be flagged as a fall, so understanding this confusion (rather than hiding the sirens) was more useful than pretending the dataset was perfectly clean.

## 3. The whisper test

A quick, informal robustness check: does the model rely on volume, or on the actual shape of distress?

Two tests were run against the trained model via `predict.py`:

1. A real scream clip, artificially quieted by 20dB and 40dB (roughly simulating "whisper-level" volume) — confidence stayed above 97% even at 40dB quieter, suggesting the model is genuinely reading the *spectral shape* of the sound, not just its loudness.
2. A real recorded human whisper (not volume-scaled — an actual quiet spoken "help me", genuine vocal cord physics) — correctly flagged as distress, but at only ~35% confidence, notably weaker than the ~97%+ confidence given to loud training-distribution screams.

Together these suggest the model has learned real spectral patterns (test 1), but the `distress` class likely skews toward loud vocalizations, so it hasn't strongly learned the acoustic signature of *quiet* fear (test 2). This is named directly in the README's Future Work section rather than glossed over.